# Evaluate Possible Outcome Metrics For CBs

In this specific EDA, we will be looking into performance metrics when CBs are matched up in <u>**MAN COVERAGE**</u> against a WR. 

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

In [ ]:
# ==============================
# Specify Position of Interest
# ==============================

position_of_interest = "CB"

<br><br/>
## Load Data
***

In [ ]:
# =============================
# Players Data
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------
players_df = (
    players_df
    .filter(pl.col("nfl_position") == position_of_interest)
)

# -----------------------------
# Only Keep Variables of Interest
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

print("Players Data:")
print(f"    - Number of Players In Position Of Interest From Rookie Cohort: {len(players_of_interest)}")
players_df

In [ ]:
# =============================
# Combine Results Data
# =============================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_results_df = (
    combine_results_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Drop Unneeded Columns
# -----------------------------

combine_results_df = (
    combine_results_df
    .drop("draft_year")
)

print("Combine Results Data:")
print(f"    - Number of Players In This Dataset: {combine_results_df.select(pl.col("nfl_id")).n_unique()}")
combine_results_df

In [ ]:
# =============================
# Combine Tracking Data
# =============================

combine_tracking_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_tracking.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Don't Keep Ball Tracking Data
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("entity_type") == "PLAYER")
)

print("Combine Tracking Data:")
print(f"    - Number of Players In This Dataset: {combine_tracking_df.select(pl.col("nfl_id")).n_unique()}")
combine_tracking_df

In [ ]:
# =============================
# Player Exposure Data
# =============================

player_exposure_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_career_successes.csv"), 
        schema_overrides = schema_map["player_career_successes"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

player_exposure_df = (
    player_exposure_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Only Keep Exposure Variables
# -----------------------------

player_exposure_df = (
    player_exposure_df
    .select(
        "nfl_id",
        "career_offensive_snaps", 
        "career_defensive_snaps",
        "career_special_teams_snaps", 
        "career_games_active"
    )
    # ------------------------------------
    # Add variable for total snaps played
    # ------------------------------------
    .with_columns(
        career_total_snaps = (
            pl.sum_horizontal(
                "career_offensive_snaps", 
                "career_defensive_snaps", 
                "career_special_teams_snaps"
            )
        )
    )
)

print("Player Exposure Data:")
print(f"    - Number of Players In This Dataset: {player_exposure_df.select(pl.col("nfl_id")).n_unique()}")
player_exposure_df

In [ ]:
# =============================
# Player-Play Data
# =============================

player_play_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["player_play"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# ------------------------------
# Only Consider Player-Plays Where 
# They're Lined Up At Position
# ------------------------------

player_play_df = (
    player_play_df
    .filter(pl.col("lined_up_position") == position_of_interest)
)

# ------------------------------
# When Analyzing CB's Only Look At Man Coverage
# ------------------------------

player_play_df = (
    player_play_df
    .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
)

# -----------------------------
# Identify Number of Players Left
# -----------------------------

num_players_with_man_coverage = player_play_df.select("nfl_id").n_unique()
perc_players_with_man_coverage = round(100 * num_players_with_man_coverage / players_df.height)

print("Player-Plays Data For CBs In Man Coverage:")
print(f"    - Number of Players: {num_players_with_man_coverage} ({perc_players_with_man_coverage}%)")
player_play_df

In [ ]:
# =============================
# Game Tracking Data
# =============================

# -----------------------------
# 2023 Game Tracking Data
# -----------------------------

game_tracking_2023_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "game_tracking_2023.csv"), 
        schema_overrides = schema_map["game_tracking"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# 2024 Game Tracking Data
# -----------------------------

game_tracking_2024_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "game_tracking_2024.csv"), 
        schema_overrides = schema_map["game_tracking"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# 2025 Game Tracking Data
# -----------------------------

game_tracking_2025_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "game_tracking_2025.csv"), 
        schema_overrides = schema_map["game_tracking"]
    )

    # ----------------------------------
    # Only Keep Players of Interest
    # ----------------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# -----------------------------
# Combine All Tracking Datasets
# -----------------------------

game_tracking_df = (
    pl.concat(
        [
            game_tracking_2023_df, 
            game_tracking_2024_df, 
            game_tracking_2025_df
        ]
    )
)

print("Game Tracking Data:")
print(f"    - Number of Players In This Dataset: {game_tracking_df.select(pl.col("nfl_id")).n_unique()}")
print(f"    - Number of Players Missing: {len(players_of_interest) - game_tracking_df.select(pl.col("nfl_id")).n_unique()}")
game_tracking_df

<br><br/>
## Evaluate Possible Outcomes
***